# Waste image classification, 2026 rebuild: DNN vs CNN vs transfer learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam843/waste-image-classification/blob/main/notebooks/waste_classification_2026.ipynb)

Originally built for DSAI4101 Deep Learning, UDST, Fall 2025 (see `original_2025_dnn.ipynb`). Rebuilt on public data in 2026.

What this notebook does, top to bottom:

1. Downloads TrashNet (MIT licence), keeps glass, metal, paper and plastic, hashes every image and checks the committed 70/15/15 split (`data_split/trashnet_split.csv`).
2. Fits two baselines: majority class and logistic regression on 64x64 pixels.
3. Trains the 2025 DNN [512,256,128], a small CNN and a frozen ResNet-18, 3 seeds each. Early stopping, LR schedule and checkpoint choice use the validation set only. The test set is used once per trained model.
4. Saves `results/results.json` and the figures in `figures/`, and exports the demo model (not committed).

How to run on Colab: Runtime > Change runtime type > T4 GPU, then Runtime > Run all. The notebook prints its own timings.
In CI it runs with `WASTE_CONFIG=smoke` (tiny synthetic images, CPU, no downloads).

In [5]:
# Setup: find (or fetch) the project code and make it importable.
import os
import subprocess
import sys
from pathlib import Path

CONFIG = os.environ.get("WASTE_CONFIG", "full")  # "full" on Colab, "smoke" in CI
REPO_URL = "https://github.com/maryam843/waste-image-classification"
IN_COLAB = "google.colab" in sys.modules


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "waste_cls").is_dir():
            return p
    return None


ROOT = find_root()
if ROOT is None and IN_COLAB:
    target = Path("/content/waste-image-classification")
    if not target.exists():
        url = REPO_URL
        try:  # only needed while the repository is private: a read-only token in Colab secrets
            from google.colab import userdata

            url = REPO_URL.replace("https://", f"https://x-access-token:{userdata.get('GH_TOKEN')}@")
        except Exception:
            pass
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(target)], capture_output=True)
        if r.returncode != 0:
            raise RuntimeError("clone failed: add a GH_TOKEN Colab secret or upload the repository zip")
        # remove the token from the saved git config
        subprocess.run(["git", "-C", str(target), "remote", "set-url", "origin", REPO_URL], check=True)
    ROOT = target
assert ROOT is not None, "run this notebook from inside the repository"
os.chdir(ROOT)
# no pip install needed: Colab already has torch, torchvision, numpy, scikit-learn, pillow and matplotlib
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
print("project:", ROOT.name, "| config:", CONFIG)

project: waste-image-classification | config: full


In [6]:
import json

from IPython.display import Image, Markdown, display

from check_readme_claims import render_table
from waste_cls.config import get_config
from waste_cls.data import prepare_data
from waste_cls.experiment import build_results, environment_info, pick_device, run_all, run_baselines, save_results
from waste_cls.models import DISPLAY_NAMES, build_model, count_parameters
from waste_cls.pipeline import export_demo_model, make_figures

cfg = get_config(CONFIG)
device = pick_device()
FULL = cfg.name == "full"
OUT = Path("results") if FULL else Path("runs/smoke")
FIG = Path("figures") if FULL else OUT / "figures"
EXPORT = Path("exports/demo_model.pt") if FULL else OUT / "demo_model.pt"
print(json.dumps(environment_info(device), indent=1))
if FULL and device == "cpu":
    print("WARNING: no GPU found. On Colab use Runtime > Change runtime type > T4 GPU.")

{
 "python": "3.13.15",
 "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
 "torch": "2.11.0+cu128",
 "torchvision": "0.26.0+cu128",
 "numpy": "2.1.3",
 "scikit_learn": "1.6.1",
 "device": "cuda",
 "gpu_name": "Tesla T4"
}


## 1. Data

TrashNet is downloaded from its GitHub repository at a pinned commit and checked against a SHA-256.
Every image is hashed. Exact duplicates stay in the same split; three duplicate pairs carry different labels
(for example the same file is in both `glass/` and `plastic/`), so those 6 files are dropped.
The split is rebuilt with seed 42 and must equal the committed CSV, otherwise this cell fails.

In [7]:
data = prepare_data(cfg, "data", "data_split/trashnet_split.csv")
print("source:", data.source, "| split hash:", data.split_hash)
print(f"{'split':6s} " + " ".join(f"{c:>8s}" for c in data.counts["train"]) + "   total")
for s, c in data.counts.items():
    print(f"{s:6s} " + " ".join(f"{v:8d}" for v in c.values()) + f"   {sum(c.values()):5d}")
print("dropped (label-conflicting duplicates):", [d["source_name"] for d in data.dropped])

source: trashnet | split hash: 479be6bb4d525c5a476b2985eb53a7d95e066dd49f247e4356a5675b0531d1d4
split     glass    metal    paper  plastic   total
train       348      286      416      336    1386
val          75       61       89       72     297
test         75       62       89       72     298
dropped (label-conflicting duplicates): ['glass/glass115.jpg', 'metal/metal91.jpg', 'glass/glass176.jpg', 'plastic/plastic152.jpg', 'glass/glass389.jpg', 'plastic/plastic332.jpg']


## 2. Baselines

Majority class (always predicts the most common training class) and logistic regression on downsampled pixels.
The regularisation strength C is picked on the validation set; the test set is scored once.

In [8]:
baselines = run_baselines(data, cfg)
for name, b in baselines.items():
    print(f"{name:14s} test acc {b['accuracy']:.3f}  macro F1 {b['macro_f1']:.3f}")
print("logistic regression val macro F1 by C:", baselines["logreg_pixels"]["val_macro_f1_by_C"])

majority       test acc 0.299  macro F1 0.115
logreg_pixels  test acc 0.510  macro F1 0.493
logistic regression val macro F1 by C: {'0.001': 0.5419890830923229, '0.01': 0.5106430094053769, '0.1': 0.525134468373462, '1.0': 0.4980662839379344}


## 3. Models

- **DNN [512,256,128]**: the 2025 network, unchanged (BatchNorm, Dropout 0.5, Adam 1e-3), no augmentation as in 2025.
- **SmallCNN**: 4 conv blocks with BatchNorm, trained from scratch, light augmentation (flip, small rotation, colour jitter).
- **ResNet-18 (frozen)**: ImageNet weights, only a new linear head is trained. Reference point for transfer learning; my full transfer-learning work is in OryxEye.

Each (model, seed) run builds a fresh model. The test set is only touched after training.

In [9]:
for name in cfg.models:
    m = build_model(name, image_size=cfg.image_size, pretrained=False)
    total, trainable = count_parameters(m), count_parameters(m, trainable_only=True)
    print(f"{DISPLAY_NAMES[name]:22s} parameters {total:>12,}  trainable {trainable:>12,}")
    del m

DNN [512,256,128]      parameters   77,237,380  trainable   77,237,380
SmallCNN               parameters      389,924  trainable      389,924
ResNet-18 (frozen)     parameters   11,178,564  trainable        2,052


In [10]:
runs, best_models = run_all(cfg, data, device, checkpoint_dir="checkpoints")

[dnn seed 0] training (augmentation=off)
  epoch   1  train loss 1.3372 f1 0.351 | val loss 1.1451 f1 0.563 *
  epoch   2  train loss 1.2140 f1 0.432 | val loss 1.0636 f1 0.583 *
  epoch   3  train loss 1.1370 f1 0.502 | val loss 1.0299 f1 0.566
  epoch   4  train loss 1.1061 f1 0.512 | val loss 0.9974 f1 0.600 *
  epoch   5  train loss 1.0437 f1 0.549 | val loss 0.9462 f1 0.582
  epoch   6  train loss 0.9800 f1 0.581 | val loss 0.9425 f1 0.615 *
  epoch   7  train loss 0.9327 f1 0.602 | val loss 0.9300 f1 0.619 *
  epoch   8  train loss 0.8700 f1 0.634 | val loss 0.9726 f1 0.633 *
  epoch   9  train loss 0.8594 f1 0.643 | val loss 0.9315 f1 0.625
  epoch  10  train loss 0.8185 f1 0.656 | val loss 0.9259 f1 0.632
  epoch  11  train loss 0.7917 f1 0.661 | val loss 0.9158 f1 0.635 *
  epoch  12  train loss 0.7723 f1 0.669 | val loss 0.9251 f1 0.649 *
  epoch  13  train loss 0.7284 f1 0.700 | val loss 0.9720 f1 0.609
  epoch  14  train loss 0.7333 f1 0.698 | val loss 0.9334 f1 0.616
  epo

[resnet18 seed 0] training (augmentation=on)
  epoch   1  train loss 1.0648 f1 0.544 | val loss 0.7790 f1 0.754 *
  epoch   2  train loss 0.7078 f1 0.734 | val loss 0.6443 f1 0.775 *
  epoch   3  train loss 0.5871 f1 0.786 | val loss 0.5558 f1 0.790 *
  epoch   4  train loss 0.5295 f1 0.801 | val loss 0.5236 f1 0.803 *
  epoch   5  train loss 0.4871 f1 0.821 | val loss 0.5131 f1 0.818 *
  epoch   6  train loss 0.4496 f1 0.841 | val loss 0.4700 f1 0.825 *
  epoch   7  train loss 0.4170 f1 0.851 | val loss 0.4611 f1 0.833 *
  epoch   8  train loss 0.4126 f1 0.856 | val loss 0.4261 f1 0.836 *
  epoch   9  train loss 0.3965 f1 0.848 | val loss 0.4556 f1 0.839 *
  epoch  10  train loss 0.3774 f1 0.860 | val loss 0.4283 f1 0.832
  epoch  11  train loss 0.3756 f1 0.858 | val loss 0.4085 f1 0.849 *
  epoch  12  train loss 0.3572 f1 0.868 | val loss 0.4028 f1 0.842
  epoch  13  train loss 0.3555 f1 0.870 | val loss 0.4055 f1 0.845
  epoch  14  train loss 0.3370 f1 0.880 | val loss 0.4320 f1 0.8

## 4. Results

The table below is rendered from `results.json` by the same function that `scripts/check_readme_claims.py` uses,
so the README can only show these numbers.

In [11]:
results = build_results(cfg, data, baselines, runs, device)
save_results(results, OUT / "results.json")
display(Markdown(render_table(results)))
print("saved", OUT / "results.json")

| Model | Test accuracy | Test macro F1 | 95 % Wilson CI for accuracy | Parameters (trainable) | Train time per run |
|---|---|---|---|---|---|
| Majority class | 0.299 | 0.115 | 0.250 to 0.353 | n/a | n/a |
| Logistic regression, 64x64 pixels | 0.510 | 0.493 | 0.454 to 0.566 | n/a | n/a |
| DNN [512,256,128] (2025 network) | 0.664 +/- 0.012 | 0.650 +/- 0.012 | 0.595 to 0.725 (widest over seeds) | 77,237,380 (77,237,380) | 81 s |
| SmallCNN (from scratch) | 0.780 +/- 0.005 | 0.775 +/- 0.005 | 0.724 to 0.828 (widest over seeds) | 389,924 (389,924) | 243 s |
| ResNet-18, frozen backbone (reference) | 0.843 +/- 0.007 | 0.841 +/- 0.007 | 0.789 to 0.885 (widest over seeds) | 11,178,564 (2,052) | 139 s |

Test set: 298 images. Networks: mean +/- sd over seeds 0, 1, 2. GPU: Tesla T4. Split hash: `479be6bb4d52`.

saved results/results.json


In [12]:
for f in make_figures(results, data, best_models, FIG, cfg.n_gradcam):
    print(f)
    display(Image(filename=str(f), width=900))

figures/learning_curves.png


[image output removed; the same figure is in figures/]


figures/confusion_matrices.png


[image output removed; the same figure is in figures/]


figures/errors_dnn.png


[image output removed; the same figure is in figures/]


... (4 outputs left out)

figures/gradcam.png


[image output removed; the same figure is in figures/]


figures/results_bar.png


[image output removed; the same figure is in figures/]


## 5. What changed since 2025

| 2025 problem | 2026 fix |
|---|---|
| Test set used for early stopping, LR schedule, checkpoint and hyperparameter choice | Separate validation split; test used once per trained model |
| Cells ran out of order; the "60-epoch experiment" continued from a trained model | One fresh model per run from a factory; notebook runs top to bottom in CI |
| Write-up named [1024,512,256,128], trained model was [512,256,128] | Architecture and parameter count written to results.json from the trained model |
| Plain MLP ignores image layout (77 M parameters, 99.8 % in the first layer) | Compared against a CNN and a frozen ResNet-18 |
| Course data.zip no longer available | Public TrashNet with a hashed, committed split |

The 2025 score (F1 0.6988 on the course's 100 test images) is not directly comparable with these numbers: different data, different split, and the 2025 number was selected on its own test set.

## 6. Export the demo model and collect the outputs

The demo model is the single run with the best **validation** macro F1 among models that support Grad-CAM.
Model files are never committed; on Colab the cell below zips what you need to download.

In [13]:
export_demo_model(results, best_models, cfg, EXPORT)
print("demo model:", results["demo_model"], "->", EXPORT)
subprocess.run([sys.executable, "scripts/prepare_hf_space.py", "--model", str(EXPORT),
                "--out", str(OUT / "hf_space_build" if not FULL else Path("hf_space_build"))], check=True)
if IN_COLAB:
    import shutil

    from google.colab import files

    shutil.make_archive("hf_space_build", "zip", "hf_space_build")
    shutil.make_archive("colab_outputs", "zip", ".", "results")
    shutil.make_archive("colab_figures", "zip", ".", "figures")
    for z in ("colab_outputs.zip", "colab_figures.zip", "hf_space_build.zip"):
        files.download(z)
    print("Also save this notebook with outputs: File > Download > Download .ipynb")

demo model: {'arch': 'resnet18', 'seed': 0, 'val_macro_f1': 0.8642412911885002, 'selected_by': 'highest validation macro F1 among Grad-CAM capable models'} -> exports/demo_model.pt


Also save this notebook with outputs: File > Download > Download .ipynb
